# ⏩ Flussi di lavoro sequenziali per agenti con Microsoft Foundry (Python)

## 📋 Tutorial avanzato di elaborazione sequenziale

Questo notebook dimostra **schemi di flussi di lavoro sequenziali** utilizzando il Microsoft Agent Framework. Imparerai a costruire pipeline di elaborazione multi-fase sofisticate in cui gli agenti vengono eseguiti in un ordine specifico, passando dati e contesto tra le fasi.

> **Nota sulla migrazione:** Questo esempio riferiva precedentemente a GitHub Models. GitHub Models è deprecato (ritirato da luglio 2026), quindi ora utilizza **Microsoft Foundry** tramite `FoundryChatClient`, che si rivolge all’API **Responses** di Azure OpenAI.

## 🎯 Obiettivi di apprendimento

### 🔄 **Schemi di elaborazione sequenziale**
- **Progettazione del flusso di lavoro lineare**: Creazione di pipeline di elaborazione passo dopo passo
- **Gestione del flusso di dati**: Passaggio di informazioni tra agenti sequenziali
- **Elaborazione a stadi**: Implementazione di checkpoint e fasi di convalida
- **Monitoraggio del progresso**: Controllo dell’esecuzione del flusso di lavoro e dei risultati intermedi

### 🏗️ **Architettura della pipeline aziendale**
- **Modellazione dei processi aziendali**: Mappatura dei processi reali nei flussi di lavoro degli agenti
- **Assicurazione della qualità**: Processi di convalida e revisione multistadio
- **Elaborazione documentale**: Analisi e trasformazione sequenziale dei documenti
- **Produzione di contenuti**: Flussi editoriali con fasi di revisione e approvazione

### 📊 **Funzionalità avanzate del flusso di lavoro**
- **Preservazione del contesto**: Mantenere lo stato attraverso le fasi del flusso di lavoro
- **Propagazione degli errori**: Gestione dei fallimenti nell’elaborazione sequenziale
- **Ottimizzazione delle prestazioni**: Schemi efficienti di esecuzione sequenziale
- **Tracce di controllo**: Tracciamento completo delle operazioni sequenziali

## ⚙️ Prerequisiti e configurazione

### 📦 **Dipendenze**
```bash
pip install agent-framework -U
```

### 🔑 **Configurazione**

Effettua il login con Azure CLI (`az login`) così che `AzureCliCredential` possa autenticarti, quindi imposta i dettagli del tuo progetto Microsoft Foundry.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

## 🏢 **Casi d’uso aziendali per flussi di lavoro sequenziali**

### 📝 **Pipeline di elaborazione documentale**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Flusso di lavoro per assicurazione della qualità** 
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Pipeline di produzione di contenuti**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Automazione dei processi aziendali**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Principi di progettazione dei flussi di lavoro sequenziali**

- **🔗 Progressione lineare**: Ogni fase dipende dall’output della fase precedente
- **📋 Gestione dello stato**: Conservare contesto e dati in tutte le fasi
- **🛡️ Gestione degli errori**: Gestione dei fallimenti in modo elegante in qualsiasi fase
- **📊 Monitoraggio del progresso**: Tracciare il completamento e le prestazioni in ogni fase
- **🔄 Riutilizzabilità degli stadi**: Progettare componenti di flusso di lavoro riutilizzabili

Costruiamo flussi di lavoro sequenziali sofisticati! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal message with an image of a
# living room. To keep the lesson focused on sequential workflow mechanics, this
# migration passes a textual description of the same scene as the workflow input.
# Agents accept a plain string, matching the basic and concurrent samples.
message = (
    "I am furnishing a modern living room and want pieces that fit a warm, "
    "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
    "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
    "Please find appropriate furniture and give the corresponding price for "
    "each piece, then produce a final purchase quote."
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
